In [ ]:
# Demo (PyTorch)

# Overview
- FER2013 dataset (face images + facial expression)
  + Goal: classifying facial expression
  + Input: a 48x48 image (PyTorch format: `(1, 48, 48)` - NCHW)
  + Output: facial expression (`angry`, `disgust`, `fear`, `happy`, `sad`, `surprise`, `neutral`)
- Defining a CNN+MLP in PyTorch
- Training and evaluating a CNN+MLP

# Import libraries

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
num_classes = 7   # angry, disgust, fear, happy, sad, surprise, neutral

# (1) Data

### Download data at https://drive.google.com/drive/folders/0BxINLo5jshCRbXpRcnl2SjJkdlk?resourcekey=0-7qr7Ro7hgkmFknsN1yParQ&usp=sharing
Dữ liệu `.csv` nằm trong `data/fer2013/` — chạy notebook từ thư mục gốc BTL03

In [ ]:
trainingset = np.loadtxt('data/fer2013/fer2013_training_onehot.csv', delimiter=',')
testingset = np.loadtxt('data/fer2013/fer2013_publictest_onehot.csv', delimiter=',')

print('trainingset shape:', trainingset.shape)
print('testingset shape :', testingset.shape)

In [ ]:
n_inputs = 2304
n_classes = 7
img_dim = 48

x_training = trainingset[:, 0:n_inputs].astype(np.float32)
y_training_oh = trainingset[:, n_inputs:n_inputs + n_classes]

x_testing = testingset[:, 0:n_inputs].astype(np.float32)
y_testing_oh = testingset[:, n_inputs:n_inputs + n_classes]

# Reshape trực tiếp về định dạng NCHW (N, 1, 48, 48) - sửa luôn lỗi deprecated expand_dims(axis=4) ở bản cũ
x_training = x_training.reshape(-1, 1, img_dim, img_dim)
x_testing = x_testing.reshape(-1, 1, img_dim, img_dim)

# Chuyển one-hot -> nhãn số nguyên (0..6) cho nn.CrossEntropyLoss
y_training = np.argmax(y_training_oh, axis=1).astype(np.int64)
y_testing = np.argmax(y_testing_oh, axis=1).astype(np.int64)

print('x_training:', x_training.shape, '| y_training:', y_training.shape)
print('x_testing :', x_testing.shape,  '| y_testing :', y_testing.shape)

# (2) Declare model

In [ ]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        # Input: (1, 48, 48)
        self.conv1_1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)    # same  -> (64, 48, 48)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=0)   # valid -> (64, 46, 46)
        self.pool1 = nn.MaxPool2d(2, 2)                              #       -> (64, 23, 23)
        self.conv_dropout1 = nn.Dropout(0.25)

        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)  # same  -> (128, 23, 23)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=0) # valid -> (128, 21, 21)
        # Giữ đúng 218 filters theo notebook Keras gốc (19,028,287 params)
        self.conv2_3 = nn.Conv2d(128, 218, kernel_size=3, padding=1) # same  -> (218, 21, 21)
        self.conv2_4 = nn.Conv2d(218, 218, kernel_size=3, padding=0) # valid -> (218, 19, 19)
        self.pool2 = nn.MaxPool2d(2, 2)                              #       -> (218, 9, 9)
        self.conv_dropout2 = nn.Dropout(0.25)

        # Flatten: 218 * 9 * 9 = 17,658
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(218 * 9 * 9, 1024)
        self.fc_dropout1 = nn.Dropout(0.5)
        self.fc3_7ways = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.conv_dropout1(self.pool1(x))

        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = F.relu(self.conv2_3(x))
        x = F.relu(self.conv2_4(x))
        x = self.conv_dropout2(self.pool2(x))

        x = self.flatten(x)
        x = self.fc_dropout1(F.relu(self.fc1(x)))
        return self.fc3_7ways(x)

model = EmotionCNN(num_classes=num_classes).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total params: {total_params:,}')  # Khớp chính xác 19,028,287 params của bản Keras

In [ ]:
try:
    from torchinfo import summary
    print(summary(model, input_size=(1, 1, 48, 48), device=device))
except ImportError:
    print('Cài đặt torchinfo nếu muốn xem bảng summary chi tiết: pip install torchinfo')

# (3) Train defined model

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            total_loss += loss.item() * bx.size(0)
            correct += (logits.argmax(dim=1) == by).sum().item()
            total += bx.size(0)
    return total_loss / total, correct / total


lr = 0.0001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

batch_size = 128
epochs = 15

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_training), torch.from_numpy(y_training)),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_testing), torch.from_numpy(y_testing)),
    batch_size=batch_size, shuffle=False
)

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

starting_time = time.time()
for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * bx.size(0)
        correct += (logits.argmax(dim=1) == by).sum().item()
        total += bx.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(f'Epoch {epoch:2d}/{epochs} - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')

print('> training time is %.4f minutes' % ((time.time() - starting_time) / 60))

# (4) Evaluate trained model

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

In [ ]:
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

def get_emotion(ohv_or_idx):
    if np.ndim(ohv_or_idx) == 0:
        indx = int(ohv_or_idx)
    else:
        indx = int(np.argmax(ohv_or_idx))
    return EMOTIONS[indx]

In [ ]:
img_indx = int(np.random.randint(0, x_testing.shape[0]))
sample = x_testing[img_indx]  # shape: (1, 48, 48)

model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(sample, axis=0)).to(device)
    pred_probs = F.softmax(model(inp), dim=1).cpu().numpy()

plt.imshow(sample[0], cmap='gray')
plt.show()
print('> testing image index: %d\n> true emotion: %s\n> predicted emotion: %s' % (
    img_indx, get_emotion(y_testing[img_indx]), get_emotion(pred_probs)
))